# Final report figures

This notebook is deliberately **analysis-only**: it reads versioned JSON artifacts from `report/figures/plot_data/`, renders lightweight Matplotlib figures on the login node, and saves each figure as PDF and PNG. It never loads a protein model, samples sequences, or submits a Slurm job.

If the preflight reports absent artifacts, copy the printed `sbatch` command into a terminal and rerun the relevant section after it completes.

In [1]:
%matplotlib inline
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / 'pyproject.toml').exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError('Run from inside the protein-design repository.')

if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from protein_design.analysis import report_data, report_figures

CONFIG = report_data.load_report_config(REPO_ROOT / 'conf/analysis/report_plots.yaml')
FIGURES_DIR = REPO_ROOT / CONFIG['output']['figures_dir']
report_figures.apply_report_style()

def render(plotter, stem, section):
    """Render only when the lightweight JSON inputs are complete."""
    if report_figures.preflight(CONFIG, [section]):
        return None
    fig = plotter(CONFIG)
    pdf, png = report_figures.save_figure(fig, FIGURES_DIR, stem)
    print(f'Saved: {pdf}\nSaved: {png}')
    return fig

print('Repository :', REPO_ROOT)
print('Artifacts :', report_data.data_dir(CONFIG))
print('Figures   :', FIGURES_DIR)

Repository : /cluster/home/gguidarini/protein-design
Artifacts : /cluster/home/gguidarini/protein-design/report/figures/plot_data
Figures   : /cluster/home/gguidarini/protein-design/report/figures


## Preflight

Check all expected JSON inputs before rendering. The command is printed rather than executed.

In [2]:
report_figures.preflight(CONFIG)

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/functional_metrics.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/preference_test_metrics.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_just_dpo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_dpo_650m.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections functional,generation,preference


[PosixPath('/cluster/home/gguidarini/protein-design/report/figures/plot_data/functional_metrics.json'),
 PosixPath('/cluster/home/gguidarini/protein-design/report/figures/plot_data/preference_test_metrics.json'),
 PosixPath('/cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json'),
 PosixPath('/cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json'),
 PosixPath('/cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_650m.json'),
 PosixPath('/cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_just_dpo_650m.json'),
 PosixPath('/cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_dpo_650m.json')]

## Evo-tuning: does antibody adaptation improve ESM2?

All panels use the same held-out DMS test definitions. CDR pseudo-perplexity is computed on ED2-M22 test; lower is better.

In [3]:
fig = render(report_figures.plot_evotune_functional, 'evotuning_heldout_spearman', 'functional')
fig

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/functional_metrics.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections functional


In [4]:
fig = render(report_figures.plot_cdr_pseudo_perplexity, 'evotuning_cdr_pseudo_perplexity', 'functional')
fig

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/functional_metrics.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections functional


## DPO: held-out preference learning

The artifact uses the original training summary only when all three requested metrics are present. Otherwise the Slurm collector recomputes the complete tuple from the checkpoint and its recovered evaluation configuration.

In [5]:
fig = render(report_figures.plot_preference_metrics, 'dpo_heldout_preference_metrics', 'preference')
fig

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/preference_test_metrics.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections preference


## Functional association and compact model table

ED2 is held out; ED5 and ED8–11 are shown as out-of-distribution generalization panels. The paired model groups share the same y-axis scale.

In [6]:
fig = render(lambda cfg: report_figures.plot_functional_groups(cfg, [['vanilla_650m', 'just_dpo_650m', 'just_lora_dpo_650m'], ['evo_650m', 'evo_dpo_650m', 'evo_lora_dpo_650m']], title=''), 'dpo_functional_association_by_base', 'functional')
fig

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/functional_metrics.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections functional


In [7]:
fig = render(report_figures.plot_all_models_functional, 'dpo_functional_association_all_models', 'functional')
fig

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/functional_metrics.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections functional


In [8]:
if report_figures.preflight(CONFIG, ['functional', 'preference']):
    fig = None
else:
    fig = report_figures.plot_all_model_table(CONFIG)
    report_figures.save_figure(fig, FIGURES_DIR, 'all_models_compact_summary')
fig

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/functional_metrics.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/preference_test_metrics.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections functional,preference


## Generation: quality, diversity, and sequence composition

Each model has a matched 5,000-sequence library for random, PSSM, Gibbs, and stochastic beam search. Native and common ESM2 likelihood views remain separate throughout.

In [9]:
fig = render(report_figures.plot_generation_quality_diversity, 'generation_quality_diversity_tradeoff', 'generation')
fig

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_just_dpo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_dpo_650m.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections generation


In [10]:
fig = render(report_figures.plot_generation_mutation_distance, 'generation_mutation_distance_by_model', 'generation')
fig

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_just_dpo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_dpo_650m.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections generation


In [11]:
for evaluator, stem in [('native', 'generation_library_summary_native_pll'), ('common', 'generation_library_summary_common_esm2_pll')]:
    fig = render(lambda cfg, evaluator=evaluator: report_figures.plot_generation_summary_table(cfg, evaluator), stem, 'generation')
    if fig is not None:
        display(fig)

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_just_dpo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_dpo_650m.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections generation
Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json
   /cluster/home/gguidarini/protein-design/report/

In [12]:
for evaluator, stem in [('native', 'generation_sequence_logos_native_pll'), ('common', 'generation_sequence_logos_common_esm2_pll')]:
    fig = render(lambda cfg, evaluator=evaluator: report_figures.plot_generation_logos(cfg, evaluator), stem, 'generation')
    if fig is not None:
        display(fig)

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_just_dpo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_dpo_650m.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections generation
Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json
   /cluster/home/gguidarini/protein-design/report/

In [13]:
for evaluator, stem in [('native', 'generation_position_jsd_native_pll'), ('common', 'generation_position_jsd_common_esm2_pll')]:
    fig = render(lambda cfg, evaluator=evaluator: report_figures.plot_generation_jsd(cfg, evaluator), stem, 'generation')
    if fig is not None:
        display(fig)

Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_just_dpo_650m.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_evo_dpo_650m.json

Run on the cluster (the notebook never submits it):
  sbatch bash_scripts/report_plot_data.sbatch --config conf/analysis/report_plots.yaml --sections generation
Missing report artifacts:
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_reference_ed2_m22_test.json
   /cluster/home/gguidarini/protein-design/report/figures/plot_data/generation_library_vanilla_650m.json
   /cluster/home/gguidarini/protein-design/report/

## Visual QA

Inspect both formats before committing a figure: labels must be readable at paper scale, model colours must remain stable, legends must not cover data, and PDF text/axes should remain vector.